# Urban air: four models, one dataset, resumable (CNN-LSTM and LSTM-only, with and without GeoCP)

Run the cells **top to bottom** (or *Save & Run All*). Every setting lives in **cell 1**.

**What it trains and compares** (the same dataset, the same test stations):

| name | CNN branch | GeoCP |
|---|---|---|
| `cnn_nogeocp` | yes | no (no calibration stations: they train the model) |
| `nocnn_nogeocp` | no (LSTM only) | no |
| `cnn_geocp` | yes | yes (calibration stations held out, intervals) |
| `nocnn_geocp` | no (LSTM only) | yes |

Optional **diagnostic** (add `"cnn_nolags"` and `"nocnn_nolags"` to `RUN_VARIANTS`): the same two models with **no PM2.5 history at all** (`data.temporal.use_pm25_lags=false`: the temporal
branch sees only the hour of day). It answers whether the morphology carries signal on its own (CNN-LSTM against LSTM-only here) or is redundant with the PM2.5 lags. No maps for these.

Comparing the two models of a row of the table (same GeoCP setting) tells you what the morphology (the CNN branch) adds; comparing the two GeoCP settings tells you what
holding the calibration stations out costs. Maps and Grad-CAM are made for the models in `MAPS_FOR` (default: `cnn_nogeocp`): the maps of **all 24 hours** of each date with an animation and a slideshow of the day
(`INFER_HOURS`), and Grad-CAM on every hour in `EXPLAIN_HOURS` (default all 24) **and** their aggregate.

**Resuming, and the 12-hour limit.** Each step looks at what already exists and skips what is done. `TIME_BUDGET_H` (cell 1, default 11) makes the notebook stop
**by itself** before Kaggle's 12-hour limit: the running stage is stopped cleanly and the notebook prints where it stopped. Training continues from its last
finished cross-validation fold. To continue:

1. *Save Version* (the whole of `/kaggle/working`, including `urban-air/model/runs/`, becomes this notebook's output).
2. In the next session, *Add Input*: pick that notebook output (and your data dataset), then run all again. Cell 3 restores the repository and the runs, and every
   finished step is skipped.
3. To redo something on purpose, put its name in `REDO` in cell 1.

| Cell | What it does |
|---|---|
| 1 | settings |
| 2 | helpers and checks |
| 3 | gets the repository (restored from a saved input, or cloned), the data, the libraries (Kaggle only) |
| 4 | for each model in `RUN_VARIANTS`: preprocess, train, report, and (for `MAPS_FOR`) maps and Grad-CAM |
| 5 | the comparison table of the models |
| 6 | shows the figures |

## 1. Settings (the only cell you edit)

In [ ]:
from pathlib import Path

ARM = "from_2023_with_preinstall"      # the dataset: data/Combined Dataset/<ARM> (this one keeps the Quezon City pre-installation readings)
PIN_PARTITION = "model/configs/pinned_partition_2026-11.json"   # the test (and calibration) stations of every result so far; None = let the seed pick them again
RASTER = "master_raster_v3.tif"        # the morphology raster in data/Morphology (it must be in your data dataset)

DATES = ["2026-11-17", "2026-10-14", "2025-07-23"]    # one set of maps per day (all INFER_HOURS); Grad-CAM explains the EXPLAIN_HOURS of each and their aggregate.
                                        # 2026-11-17: the most recent full day. Its readings are thin in some hours: only 1 to 2 stations report at 12:00, 17:00 and 21:00,
                                        # so the history of those hours comes from far away and the map is less reliable there. 2026-11-18 cannot be used: the data ends at 19:00 that day (a 24-hour day is skipped).
                                        # 2026-10-14 and 2025-07-23 were mapped before: 10-14 is an OpenAQ-only day (42, 25, 43, 43 stations at 08, 12, 17, 21), 07-23 the richest day (65 to 78 stations, Quezon City included).

#RUN_VARIANTS = ["cnn_nogeocp", "nocnn_nogeocp", "cnn_geocp", "nocnn_geocp"]                        
RUN_VARIANTS = ["cnn_nogeocp"]   # the models to train, in this order (the most important first, in case time runs out). Any subset of these four; the optional diagnostic pair is "cnn_nolags", "nocnn_nolags" (no PM2.5 history, see the first cell)
MAPS_FOR = ["cnn_nogeocp"]              # the models that also get maps and Grad-CAM (they need the CNN branch). Add "cnn_geocp" to map that one too (with its GeoCP intervals)
TIME_BUDGET_H = 11.0                    # stop by itself after this many hours (Kaggle ends a session at 12). The running stage is stopped cleanly; Save Version, add it as an Input, run again to continue
INFER_HOURS = list(range(24))          # the hours of each date that get a map: all 24 (default) = the day as an animation and a slideshow. [8, 12, 17, 21] = the four peak hours, 6 times faster
EXPLAIN_FIRST = [8, 12, 17, 21]         # explained FIRST, for every date, so the four peak hours are done even if time runs out; then the rest of EXPLAIN_HOURS (the aggregate is then made again over all of them). [] = no first pass
EXPLAIN_HOURS = list(range(24))         # the hours Grad-CAM explains: all 24 (default; each hour is slow, so the whole day takes 6x the four-hour one). [8, 12, 17, 21] = the four peak hours. Among INFER_HOURS. Their aggregate is made too
SHOW_HOURS = list(range(24))            # which explained hours the figures cell (6) displays one by one (all of them are saved on disk; 24 x 2 figures would flood the notebook)
STRIDE = 11                             # 11 = 110 m cells, a quick look; 5 = 50 m (slower); 1 = the full 10 m grid (hours on a CPU)
SKIP_DATES_OUTSIDE_DATA = True          # True: a date with no readings behind it is skipped with a message. False: try it anyway (it stops with the reason)

REDO = []                               # steps to run again although their output exists: any of "preprocess", "train", "report", "infer", "explain"
                                        # (redoing "preprocess" redoes the later steps too: a model only fits the data it was trained on)

DATA = Path("/kaggle/input/datasets/patrickhansperez/urban-air-data-recent")                    # your data dataset: the exact path shown in the Input panel
GITHUB = "https://github.com/Patthethicc/urban-air.git"        # the code, used when no saved repository is attached
RETRAIN_IF_CONFIG_CHANGED = True       # True: a saved run trained with other model/training settings than configs/*.yaml now has is NOT reused: the model is trained again
                                        # (preprocessing is kept). False: always reuse the newest saved run, whatever settings it was trained with
REFRESH_CODE = False                    # True: after restoring a saved repository, replace its code (not its runs) by the latest from GITHUB

## 2. Helpers and checks

In [ ]:
import json, os, shutil, subprocess, sys, time, traceback
import numpy as np, pandas as pd

T0 = time.time()                         # the clock of the time budget starts here (cell 2); a new session starts it again
ON_KAGGLE = Path("/kaggle/working").exists()
INPUT = Path("/kaggle/input")
if ON_KAGGLE:
    REPO = Path("/kaggle/working/urban-air")
else:                                    # a local checkout: open the notebook from the repository root
    REPO = Path.cwd() if (Path.cwd() / "model").is_dir() else Path.cwd().parent
MODEL = REPO / "model"
OUT = Path("/kaggle/working") if ON_KAGGLE else REPO       # where the comparison table is written
STEPS = ("preprocess", "train", "report", "infer", "explain")
EXPLAIN_LABELS = tuple(f"{h:02d}:00" for h in EXPLAIN_HOURS)       # "08:00" ... the hours Grad-CAM explains, as the pipeline names them
SHOW_LABELS = tuple(l for l in (f"{h:02d}:00" for h in SHOW_HOURS) if l in EXPLAIN_LABELS)   # the ones the figures cell shows one by one
VARIANTS = {                             # name -> the two switches that make it
    "cnn_nogeocp":   {"cnn": True,  "geocp": False, "label": "CNN-LSTM, no GeoCP"},
    "nocnn_nogeocp": {"cnn": False, "geocp": False, "label": "LSTM only (no CNN), no GeoCP"},
    "cnn_geocp":     {"cnn": True,  "geocp": True,  "label": "CNN-LSTM, with GeoCP"},
    "nocnn_geocp":   {"cnn": False, "geocp": True,  "label": "LSTM only (no CNN), with GeoCP"},
    # the diagnostic: no PM2.5 history (no lags, no spatial lag), only the hour of day + the morphology. GeoCP off. Opt-in through RUN_VARIANTS, never mapped
    "cnn_nolags":    {"cnn": True,  "geocp": False, "lags": False, "label": "CNN-LSTM, NO PM2.5 history (diagnostic)"},
    "nocnn_nolags":  {"cnn": False, "geocp": False, "lags": False, "label": "LSTM only, NO PM2.5 history (diagnostic)"},
}
VARIANT = "cnn_nogeocp"                  # the model the helpers work on; cell 4 changes it as it goes
MIN_STEP_H = 0.1                         # a step is not started with less than this left of the time budget


class OutOfTime(Exception):
    """The time budget is used up: stop cleanly so that Save Version keeps everything."""


def need(*names):
    """A clear message instead of a NameError at the end of a long run: the names a cell relies on must exist."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"{', '.join(missing)} not defined: run the cells above in order (1 settings, 2 helpers, 3 setup) first")


def check_settings():
    need("ARM", "RASTER", "DATES", "INFER_HOURS", "EXPLAIN_FIRST", "EXPLAIN_HOURS", "SHOW_HOURS", "STRIDE", "SKIP_DATES_OUTSIDE_DATA", "REDO", "DATA", "GITHUB", "REFRESH_CODE", "RETRAIN_IF_CONFIG_CHANGED",
         "RUN_VARIANTS", "MAPS_FOR", "TIME_BUDGET_H")
    global REDO
    for d in DATES:
        pd.Timestamp(d)                  # a mistyped date fails here, in the first seconds, not after hours of training
    if not INFER_HOURS or any(not isinstance(h, int) or not 0 <= h <= 23 for h in INFER_HOURS) or len(set(INFER_HOURS)) != len(INFER_HOURS):
        raise ValueError(f"INFER_HOURS must be distinct whole hours from 0 to 23 (all 24: list(range(24))), got {INFER_HOURS!r}")
    if not set(EXPLAIN_FIRST) <= set(EXPLAIN_HOURS):
        raise ValueError(f"EXPLAIN_FIRST {EXPLAIN_FIRST!r} must be hours that are also in EXPLAIN_HOURS {EXPLAIN_HOURS!r}")
    if not EXPLAIN_HOURS or not set(EXPLAIN_HOURS) <= set(INFER_HOURS):
        raise ValueError(f"EXPLAIN_HOURS {EXPLAIN_HOURS!r} must be a non-empty subset of INFER_HOURS {INFER_HOURS!r}: Grad-CAM explains hours that have a map")
    if not isinstance(STRIDE, int) or STRIDE < 1 or STRIDE % 2 == 0:
        raise ValueError(f"STRIDE must be an odd number (1, 3, 5, 11, 21 ...), got {STRIDE!r}")
    unknown = [r for r in REDO if r not in STEPS]
    if unknown:
        raise ValueError(f"REDO has unknown steps {unknown}; use any of {STEPS}")
    bad = [v for v in list(RUN_VARIANTS) + list(MAPS_FOR) if v not in VARIANTS]
    if bad:
        raise ValueError(f"unknown model(s) {bad}; use any of {list(VARIANTS)}")
    nolags = [v for v in MAPS_FOR if not VARIANTS[v].get("lags", True)]
    if nolags:
        raise ValueError(f"MAPS_FOR {nolags} are the no-PM2.5-history diagnostic: they are not accurate enough to map, leave them out of MAPS_FOR")
    no_cnn = [v for v in MAPS_FOR if not VARIANTS[v]["cnn"]]
    if no_cnn:
        raise ValueError(f"MAPS_FOR {no_cnn} have no CNN branch: maps and Grad-CAM need one")
    if not TIME_BUDGET_H > 0:
        raise ValueError(f"TIME_BUDGET_H must be above 0, got {TIME_BUDGET_H!r}")
    if "preprocess" in REDO:             # a model only fits the data it was trained on: redo everything after it
        REDO = sorted(set(REDO) | set(STEPS))


def left_h():
    """Hours left of the time budget."""
    return TIME_BUDGET_H - (time.time() - T0) / 3600


def legacy_geocp_ok():
    """An earlier version of this notebook trained with GeoCP into runs/<ARM>/{processed,checkpoints,snapshots}: reuse it as the `cnn_geocp` model
    (it has calibration stations) instead of training it again."""
    info = MODEL / "runs" / ARM / "processed" / "preprocess_info.json"
    return info.is_file() and json.loads(info.read_text(encoding="utf-8")).get("stations_per_partition", {}).get("calibration", 0) > 0


def folder(kind, variant=None):
    """The folder (under runs/<ARM>/) a model keeps its `kind` in. Preprocessing is shared by the models with the same GeoCP setting (the split
    is the same); checkpoints and snapshots belong to one model."""
    v = variant or VARIANT
    geocp = VARIANTS[v]["geocp"]
    if kind == "processed":
        if not VARIANTS[v].get("lags", True):
            return "processed_nolags"                    # the diagnostic pair shares one preprocessing (different input channels than the others)
        return "processed" if geocp and legacy_geocp_ok() else f"processed_{'geocp' if geocp else 'nogeocp'}"
    if kind in ("checkpoints", "snapshots"):
        legacy = v == "cnn_geocp" and legacy_geocp_ok() and not (MODEL / "runs" / ARM / f"checkpoints_{v}").exists()
        return kind if legacy else f"{kind}_{v}"
    return kind


def pinned_stations():
    """The fixed test/calibration stations as overrides (so a longer or updated dataset is scored on the same stations); [] when PIN_PARTITION is None."""
    if not PIN_PARTITION:
        return {}
    d = json.loads((REPO / PIN_PARTITION).read_text())
    return {"data.split.test_stations": json.dumps(sorted(d["test"])), "data.split.calibration_stations": json.dumps(sorted(d["calibration"]))}


def overrides(extra=()):
    """Every stage needs the same overrides (the paths of this model and its two switches), so one function builds them."""
    spec = VARIANTS[VARIANT]
    dirs = {f"data.paths.{k}_dir": f"runs/{ARM}/{folder(k)}" for k in ("processed", "checkpoints", "snapshots", "tuning")}
    paths = {"data.paths.master_raster": f"../data/Morphology/{RASTER}",
             "data.paths.sensor_locations_csv": f"../data/Combined Dataset/{ARM}/locations.csv",
             "data.paths.sensor_csv_dir": f"../data/Combined Dataset/{ARM}/sensors"}
    hours = {"inference.snapshot_hours": json.dumps(sorted(INFER_HOURS)), "inference.explain_hours": json.dumps(sorted(EXPLAIN_HOURS))}
    kv = {**pinned_stations(), **hours, "data.split.cv_assignment": "by_station", "geocp.enabled": "true" if spec["geocp"] else "false",
          "model.spatial_cnn.enabled": "true" if spec["cnn"] else "false",
          "data.temporal.use_pm25_lags": "true" if spec.get("lags", True) else "false", **paths, **dirs}
    return [x for k, v in kv.items() for x in ("--override", f"{k}={v}")] + list(extra)


def run(stage, *flags, extra=()):
    """One pipeline stage in its own process. The stage itself is stopped when the time budget is used up (exit code 124 -> OutOfTime)."""
    if left_h() < MIN_STEP_H:
        raise OutOfTime(f"before {stage}")
    budget = ["--override", f"train.runtime.max_hours={left_h():.3f}"]
    cmd = [sys.executable, "-u", "run_pipeline.py", "--stage", stage, *overrides(extra), *budget, *flags]
    print("$", " ".join(cmd), flush=True)
    t = time.time()
    code = subprocess.run(cmd, cwd=MODEL).returncode
    if code == 124:
        raise OutOfTime(f"during {stage}")
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)                           # stops the cell if a stage fails
    print(f"-- {stage}: {time.time() - t:.0f} s", flush=True)


def have(kind, *parts):
    """Does runs/<ARM>/<folder of this model's `kind`>/<parts...> exist?"""
    return (MODEL / "runs" / ARM / folder(kind)).joinpath(*parts).exists()


def infer_done(run_id, day):
    """Is the map step of this date finished FOR THE HOURS NOW IN INFER_HOURS? (A date mapped earlier with four hours is not: the day is made again.)"""
    f = MODEL / "runs" / ARM / folder("snapshots") / run_id / "surface" / day / "output_summary.json"
    if not f.is_file():
        return False
    hours = sorted(int(s["hour"][:2]) for s in json.loads(f.read_text(encoding="utf-8")).get("snapshots", []))
    return hours == sorted(INFER_HOURS)


def explain_done(run_id, day, hours):
    """Is Grad-CAM finished for exactly these hours of the date? Every hour has its own folder, and the aggregate of the day (explain/<day>_all) was written
    after the last of them (so an aggregate made over the first four hours does not count as finished when all 24 are asked for)."""
    root = MODEL / "runs" / ARM / folder("snapshots") / run_id / "explain"
    parts = [root / f"{day}_{h:02d}00" / "gradcam_summary.json" for h in hours]
    if not all(f.is_file() for f in parts):
        return False
    if len(hours) < 2:
        return True
    agg = root / f"{day}_all" / "gradcam_summary.json"
    return agg.is_file() and agg.stat().st_mtime >= max(f.stat().st_mtime for f in parts)


def runs_of():
    """Finished training runs of this model (run_info.json says completed), oldest first."""
    out = []
    for info in sorted((MODEL / "runs" / ARM / folder("checkpoints")).glob("*/run_info.json")):
        if json.loads(info.read_text(encoding="utf-8")).get("status") == "completed":
            out.append(info.parent.name)
    return out


def config_changes(run_id):
    """Settings a saved run was trained with that differ from configs/*.yaml now (model, epochs, batch size, optimizer, early stopping).
    The CNN switch is the one of this model. A run without a saved copy of its settings cannot be checked: it returns []."""
    import yaml
    info = json.loads((MODEL / "runs" / ARM / folder("checkpoints") / run_id / "run_info.json").read_text(encoding="utf-8"))
    saved = info.get("hyperparameters")
    if not saved:
        return []
    now = {"model": yaml.safe_load((MODEL / "configs" / "model.yaml").read_text(encoding="utf-8")),
           "train": yaml.safe_load((MODEL / "configs" / "train.yaml").read_text(encoding="utf-8"))}
    now["model"].setdefault("spatial_cnn", {})["enabled"] = VARIANTS[VARIANT]["cnn"]

    def flat(d, prefix=""):
        for k, v in d.items():
            if isinstance(v, dict):
                yield from flat(v, f"{prefix}{k}.")
            else:
                yield f"{prefix}{k}", v
    keep = ("model.", "train.epochs", "train.batch_size", "train.optimizer.", "train.early_stopping_patience", "train.loss")
    a, b = dict(flat(saved)), dict(flat(now))
    return [f"{k}: {a.get(k)!r} -> {b.get(k)!r}" for k in sorted(set(a) | set(b)) if k.startswith(keep) and a.get(k) != b.get(k)]


def step(name, done, *flags, extra=()):
    """Run a pipeline stage unless its output already exists (and it is not in REDO)."""
    if done and name not in REDO:
        print(f"skip {name}: already done")
        return False
    run(name, *flags, extra=extra)
    return True


def status():
    """A line per model: what is finished (preprocess / train / report / maps)."""
    global VARIANT
    keep, rows = VARIANT, []
    for v in RUN_VARIANTS:
        VARIANT = v
        runs = runs_of()
        rid = runs[-1] if runs else None
        rows.append({"model": v, "preprocess": have("processed", "preprocess_info.json"), "train": bool(runs),
                     "report": bool(rid and have("snapshots", rid, "SUMMARY.md")),
                     "maps": (bool(rid) and all(explain_done(rid, d.replace('-', ''), sorted(EXPLAIN_HOURS)) for d in DATES))
                          if v in MAPS_FOR else None})
    VARIANT = keep
    df = pd.DataFrame(rows).set_index("model").apply(lambda col: col.map(lambda x: "n/a" if x is None else "done" if x else "-"))
    print(df.to_string())
    print(f"time used: {(time.time() - T0) / 3600:.1f} h of {TIME_BUDGET_H} h")


check_settings()
print("settings ok | dataset:", ARM, "| raster:", RASTER, "| dates:", DATES, "| models:", RUN_VARIANTS, "| repository:", REPO)

## 3. Get the repository, the data and the libraries
On Kaggle: if a saved repository is attached as an Input (a previous version of this notebook), it is restored, runs and all; otherwise the code is
cloned from GitHub. The data is copied from your data dataset (it is found even if the upload added a folder level). Locally this cell only checks that
everything is where it should be.

In [ ]:
check_settings()


def find_saved_repo():
    """A repository saved by an earlier session: any attached input that contains model/run_pipeline.py."""
    if INPUT.exists():
        for p in INPUT.glob("**/model/run_pipeline.py"):
            return p.parent.parent
    return None


def clone(into):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB, str(into)], check=True)


if ON_KAGGLE:
    if not (MODEL / "run_pipeline.py").is_file():
        saved = find_saved_repo()
        if saved is not None:
            shutil.copytree(saved, REPO, dirs_exist_ok=True)
            n = len(list((MODEL / "runs").glob("*/checkpoints*/*/run_info.json"))) if (MODEL / "runs").exists() else 0
            print(f"restored the repository from {saved} ({n} saved training run(s) found)")
        else:
            clone(REPO)
            print("cloned the repository from GitHub (no saved repository attached)")
    else:
        print("the repository is already in /kaggle/working")
    if REFRESH_CODE:                     # the latest code from GitHub; the saved runs and data are left as they are
        fresh = REPO.parent / "_fresh_code"
        shutil.rmtree(fresh, ignore_errors=True)
        clone(fresh)
        shutil.copytree(fresh / "model", MODEL, dirs_exist_ok=True, ignore=shutil.ignore_patterns("runs", "__pycache__", ".env"))
        shutil.rmtree(fresh, ignore_errors=True)
        print("code refreshed from GitHub")

    raster = REPO / "data" / "Morphology" / RASTER
    if not raster.is_file():             # the data is not in place yet: find it under DATA wherever the upload put it
        found = list(DATA.rglob(RASTER)) if DATA.exists() else []
        if found:
            src = found[0].parent.parent                      # the folder that holds Morphology/, Combined Dataset/, ...
            for part in (Path("Morphology"), Path("Boundaries"), Path("Combined Dataset") / ARM):
                if (src / part).is_dir():
                    shutil.copytree(src / part, REPO / "data" / part, dirs_exist_ok=True)
            print("data copied from", src)
        else:
            zips = list(DATA.rglob("*.zip")) if DATA.exists() else []
            if zips:                                          # Kaggle left an uploaded zip packed: its top-level `data/` folder lands in REPO
                import zipfile
                zipfile.ZipFile(zips[0]).extractall(REPO)
                print("data unzipped from", zips[0])

    need_pip = {"skopt": "scikit-optimize", "libpysal": "libpysal", "esda": "esda", "rasterio": "rasterio", "geopandas": "geopandas",
                "shapely": "shapely", "pyproj": "pyproj", "affine": "affine", "yaml": "pyyaml", "sklearn": "scikit-learn", "contextily": "contextily"}
    import importlib.util
    missing = [pip for mod, pip in need_pip.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("installing:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    old_esda = "import inspect; from esda.moran import Moran_Local as M; raise SystemExit(0 if 'alternative' in inspect.signature(M.__init__).parameters else 1)"
    if subprocess.run([sys.executable, "-c", old_esda]).returncode != 0:      # the spatial-bias check needs the two-sided LISA option
        print("esda is too old: upgrading")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "esda", "libpysal"], check=True)

# the checks that stop a run in its first seconds instead of after hours
problems = []
if not (MODEL / "run_pipeline.py").is_file():
    problems.append(f"{MODEL / 'run_pipeline.py'} is missing: no repository (attach a saved one, or check GITHUB and the internet switch)")
if not (REPO / "data" / "Morphology" / RASTER).is_file():
    problems.append(f"data/Morphology/{RASTER} is missing: put it in the Morphology folder of your data dataset (DATA = {DATA})")
if not (REPO / "data" / "Combined Dataset" / ARM / "locations.csv").is_file():
    problems.append(f"data/Combined Dataset/{ARM}/locations.csv is missing: check ARM and your data dataset")
if problems:
    raise SystemExit("cannot start:\n  - " + "\n  - ".join(problems))
print("ready | state of each model:")
status()

## 4. Train, report and map each model
For each model in `RUN_VARIANTS`, in order: preprocess (shared by the models with the same GeoCP setting), train, report; the models in `MAPS_FOR` then get the maps and Grad-CAM of every date in `DATES`.
Each step is skipped if its output is already there. Training uses the values in `model/configs/` with seed 42 and continues from the last finished fold of a stopped run.

When `TIME_BUDGET_H` is used up the notebook stops cleanly and prints what to do (Save Version, add the output as an Input, run again).

A date is mapped only if the data has readings to build its history from (the three hours before each snapshot). A 2025 date uses every training station; from 2026 on the history comes from the nearest OpenAQ stations only.
The maps cover **every hour in `INFER_HOURS`** (default: all 24, a day you can play as an animation or a slideshow; `[8, 12, 17, 21]` gives the four peak hours and is six times faster).
Grad-CAM explains the hours in `EXPLAIN_HOURS` (default all 24; slow, so `[8, 12, 17, 21]` is the quick choice) and then their **aggregate** (`explain/<day>_all/`: the figures of the hours pooled, a by-hour chart, and the hot and cold cells of the average of those hours).
A failed date is printed and does **not** stop the notebook.

Grad-CAM runs in two passes: the `EXPLAIN_FIRST` hours (8, 12, 17, 21) of every date first, then the remaining `EXPLAIN_HOURS`; the aggregate of the day is made again over all explained hours at the end. A finished hour is never explained twice, so a stopped run continues where it left off.

In [ ]:
check_settings()
MAPPED = {}                              # model -> the dates that have maps and Grad-CAM
RUN_IDS = {}                             # model -> its finished training run


def map_dates(run_id):
    """Maps (infer: every hour in INFER_HOURS, with the animation) and Grad-CAM (the EXPLAIN_HOURS + their aggregate) of every date in DATES for the current model."""
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["CARTO_API_KEY"] = UserSecretsClient().get_secret("CARTO_API_KEY")
        print("CARTO_API_KEY set from Kaggle Secrets")
    except Exception as e:                # not on Kaggle, secret not attached, or a different label
        print(f"CARTO_API_KEY not set from Kaggle Secrets ({type(e).__name__}); the maps use the offline basemap unless model/.env has a key")
    z = np.load(MODEL / "runs" / ARM / folder("processed") / "pm25_hourly.npz")
    data_start = pd.Timestamp(str(z["start"]))
    data_end = data_start + pd.Timedelta(hours=len(z["values"]) - 1)             # the last hour with a reading
    print(f"the data runs from {data_start:%Y-%m-%d %H:%M} to {data_end:%Y-%m-%d %H:%M}")
    stride = ["--override", f"inference.stride_px={STRIDE}"]
    redo_explain = ["--override", "output.explain.redo=true"] if "explain" in REDO else []
    first = sorted(h for h in EXPLAIN_FIRST if h in EXPLAIN_HOURS)
    phases = ([first] if first and set(first) != set(EXPLAIN_HOURS) else []) + [sorted(EXPLAIN_HOURS)]      # the peak hours first, then everything
    usable = []
    for date in DATES:
        day = date.replace("-", "")
        first_h, last_h = pd.Timestamp(f"{date} {min(INFER_HOURS):02d}:00"), pd.Timestamp(f"{date} {max(INFER_HOURS):02d}:00")      # the first and last hours that get a map
        outside = first_h < data_start + pd.Timedelta(hours=3) or last_h > data_end + pd.Timedelta(hours=1)
        if outside and SKIP_DATES_OUTSIDE_DATA:
            print(f"{date}: skipped, it lies outside the data, so there are no readings to build the history from")
            continue
        try:
            step("infer", infer_done(run_id, day), "--run-id", run_id, "--date", date, extra=stride)
            usable.append(date)
        except subprocess.CalledProcessError:                 # the pipeline's own message is printed just above; go on with the next date
            print(f"{date}: the pipeline stopped (reason above), no maps for this date")
    failed = set()
    for hours in phases:                                       # phase 1: the EXPLAIN_FIRST hours of every date; phase 2: all EXPLAIN_HOURS (finished hours are reused)
        print(f"Grad-CAM, hours {hours}" + (" (first pass)" if hours is not phases[-1] else " (all hours; the aggregate is made again over all of them)"))
        for date in usable:
            if date in failed:
                continue
            day = date.replace("-", "")
            try:
                step("explain", explain_done(run_id, day, hours), "--run-id", run_id, "--date", date,
                     extra=stride + redo_explain + ["--override", f"inference.explain_hours={json.dumps(hours)}"])
            except subprocess.CalledProcessError:
                failed.add(date)
                print(f"{date}: Grad-CAM stopped (reason above), no Grad-CAM for the rest of this date")
    mapped = [d for d in usable if d not in failed]
    print("mapped:", mapped or "none")
    return mapped


def run_variant(v):
    global VARIANT
    VARIANT = v
    spec = VARIANTS[v]
    print("=" * 78 + f"\n{v}: {spec['label']}  ({left_h():.1f} h of the time budget left)\n" + "=" * 78)
    step("preprocess", have("processed", "scalers.json") and have("processed", "preprocess_info.json"))
    stale = config_changes(runs_of()[-1]) if runs_of() and RETRAIN_IF_CONFIG_CHANGED else []
    if stale:
        print("the newest saved run was trained with other settings than configs/*.yaml now has, so it is not reused:")
        print("\n".join("  " + s for s in stale[:12]))
    if "train" in REDO or not runs_of() or stale:
        run("train", extra=["--override", "train.resume=true"])                  # continues a run that was stopped, or starts a new one
    else:
        print("skip train: a finished run exists" + ("" if RETRAIN_IF_CONFIG_CHANGED else " (not checked against the current settings)"))
    RUN_IDS[v] = runs_of()[-1]                                                  # the newest finished run
    print("run:", RUN_IDS[v])
    step("report", have("snapshots", RUN_IDS[v], "SUMMARY.md"), "--run-id", RUN_IDS[v])
    summary = MODEL / "runs" / ARM / folder("snapshots") / RUN_IDS[v] / "SUMMARY.md"
    print(summary.read_text(encoding="utf-8")[:1800])
    if v in MAPS_FOR:
        MAPPED[v] = map_dates(RUN_IDS[v])


try:
    for v in RUN_VARIANTS:
        run_variant(v)
    print("\nevery model is finished")
except OutOfTime as stop:
    print(f"\nTIME BUDGET REACHED ({stop}) after {(time.time() - T0) / 3600:.1f} h. Nothing is lost: finished steps and finished training folds are saved.")
    print("To continue: Save Version, then in a new session Add Input (this notebook's output + your data), and run all again.")
except Exception:
    traceback.print_exc()
    print("A step failed (everything finished so far is saved). Fix the cause above and run this cell again: finished steps are skipped.")
finally:
    status()

## 5. The models side by side
Test R², RMSE and MAE on the same test stations, the cross-validation R², and the R² the CNN branch adds (CNN-LSTM minus LSTM-only) for each GeoCP setting.
With GeoCP the calibration stations are held out of training, without it they train the model, so compare the CNN effect **within** a GeoCP setting.

In [ ]:
try:
    need("RUN_VARIANTS")
    rows = []
    keep = VARIANT
    for v in RUN_VARIANTS:
        VARIANT = v
        runs = runs_of()
        if not runs:
            continue
        metrics = MODEL / "runs" / ARM / folder("snapshots") / runs[-1] / "metrics_summary.json"
        if not metrics.is_file():
            continue
        m = json.loads(metrics.read_text(encoding="utf-8"))["accuracy"]["measures"]
        t, cv = m["test"], m.get("cross_validation", {})
        rows.append({"model": v, "CNN": VARIANTS[v]["cnn"], "GeoCP": VARIANTS[v]["geocp"], "PM2.5 history": VARIANTS[v].get("lags", True), "run": runs[-1],
                     "test stations": t["stations"], "CV R2": cv.get("r2"), "test R2": t["r2"], "test RMSE": t["rmse"], "test MAE": t["mae"],
                     "test bias": t.get("bias")})
    VARIANT = keep
    table = pd.DataFrame(rows)
    if len(table):
        gain = []
        for _, r in table.iterrows():
            twin = table[(table["GeoCP"] == r["GeoCP"]) & (table["PM2.5 history"] == r["PM2.5 history"]) & (~table["CNN"])]
            gain.append(float(r["test R2"] - twin["test R2"].iloc[0]) if r["CNN"] and len(twin) else None)
        table["R2 gained by the CNN"] = gain
        pd.set_option("display.width", 200)
        print(table.round(3).to_string(index=False))
        table.round(4).to_csv(OUT / "model_comparison.csv", index=False)
        print("written:", OUT / "model_comparison.csv")
    else:
        print("no finished model with a report yet")
except Exception:
    traceback.print_exc()
    print("The comparison failed; each model's own report is runs/<dataset>/snapshots_<model>/<run>/SUMMARY.md")

## 6. Show the figures

In [ ]:
try:
    need("MAPPED")
    from IPython.display import Image, Markdown, display

    def show(path, title, width=900):
        """Display one figure, or say which file is missing (e.g. a map style that was switched off)."""
        if path.is_file():
            display(Markdown(f"**{title}**"))
            display(Image(filename=str(path), width=width))
        else:
            print("missing:", path)

    for v, dates in MAPPED.items():
        VARIANT = v
        snap = MODEL / "runs" / ARM / folder("snapshots") / RUN_IDS[v]
        display(Markdown(f"# {VARIANTS[v]['label']} (run {RUN_IDS[v]})"))
        for date in dates:
            day = date.replace("-", "")
            maps = snap / "surface" / day / "maps"
            allhours = snap / "explain" / f"{day}_all"
            display(Markdown(f"## {date}"))
            show(maps / "continuous_panels.png", f"PM2.5 prediction, {date}: continuous scale ({len(INFER_HOURS)} hours)")
            show(maps / "continuous_animation.gif", f"The day as an animation, {date} (one frame per hour, one colour scale). Also in maps/: continuous_slideshow.html (play, pause, slider) and .mp4 if imageio-ffmpeg is installed", width=700)
            show(maps / "daily_panels.png", f"PM2.5 prediction, {date}: Philippine AQI bands")
            show(maps / "continuous_daily_mean.png", f"Mean of the {len(INFER_HOURS)} hours, {date}: continuous scale", width=600)
            display(Markdown(f"### Grad-CAM, {date}: the {len(EXPLAIN_HOURS)} explained hours together"))
            show(allhours / "gradcam_map.png", "Where the hot (red) and cold (blue) cells of the average of the explained hours are", width=700)
            show(allhours / "gradcam_hot_cells.png", f"Hot cells: Grad-CAM averaged over the {len(EXPLAIN_HOURS)} explained hours")
            show(allhours / "gradcam_cold_cells.png", f"Cold cells: Grad-CAM averaged over the {len(EXPLAIN_HOURS)} explained hours")
            show(allhours / "gradcam_band_profile.png", "What the CNN attends to: hot against cold cells, all hours pooled", width=800)
            show(allhours / "gradcam_band_swarm.png", "Grad-CAM attention score per band (SHAP-style), all hours pooled", width=700)
            show(allhours / "gradcam_band_importance.png", "Mean Grad-CAM attention per band, all hours pooled", width=700)
            show(allhours / "gradcam_band_by_hour.png", "Mean Grad-CAM attention per band, by hour", width=700)
            show(allhours / "band_ablation.png", "How far each band moves the prediction, all hours pooled", width=700)
            show(allhours / "band_ablation_swarm.png", "Prediction change when a band is swapped, all hours pooled", width=700)
            for hour in SHOW_LABELS:
                one = snap / "explain" / f"{day}_{hour.replace(':', '')}"
                display(Markdown(f"### Grad-CAM, {date} {hour}"))
                show(one / "gradcam_map.png", f"Where the explained hot (red) and cold (blue) cells are, {hour}", width=600)
                show(one / "gradcam_band_profile.png", f"What the CNN attends to: hot against cold cells, {hour}", width=700)
            show(snap / "explain" / f"{day}_{EXPLAIN_LABELS[0].replace(':', '')}" / "morphology_sheet.png", "Morphology of the NCR: the five bands the CNN branch sees")
    print("To continue in another session: Save Version, then add this notebook's output as an Input; cell 3 restores it.")
except Exception:
    traceback.print_exc()
    print("Showing the figures failed; the files are under model/runs/<dataset>/snapshots_<model>/<run>/ (surface/ and explain/)")